# Lab 6b — Bamburgh Castle Reconstruction with gsplat
### YouTube Video → COLMAP / GLOMAP → gsplat Training → `.splat`

**Course**: 3D Gaussian Splatting  
**Instructor**: Nicholas McCarty (Upskilled Consulting)  
**Runtime**: ~55 min on Colab T4 (video download + SfM + 30k-iteration training + eval)

---

## Why gsplat?

The original `gaussian-splatting` repository from Inria/MPII is **research-only** — commercial use requires explicit written consent from Inria (see the license reading in Module 1). [gsplat](https://github.com/nerfstudio-project/gsplat) is a community reimplementation maintained by the nerfstudio team under the **MIT license**, meaning it can be used freely in commercial products.

Functionally, gsplat is a drop-in replacement for the original training loop:
- Takes the same COLMAP-format input (`images/` + `sparse/0/`)
- Outputs the same 59-property `.ply` schema
- Converts cleanly to `.splat` for WebGL viewers
- Adds built-in per-step PSNR/SSIM/LPIPS evaluation

| Property | original `gaussian-splatting` | gsplat |
|---|---|---|
| License | Non-commercial (Inria/MPII) | MIT |
| Input format | COLMAP | COLMAP |
| Output format | `.ply` (59 props) | `.ply` (59 props) |
| Built-in eval | No | Yes (PSNR/SSIM/LPIPS) |
| Sparse Adam | Via fork | Built-in |

This notebook runs the identical Bamburgh Castle pipeline as Lab 6, replacing only the training step.

## §0 — Install Dependencies

We use the same pre-built COLMAP and GLOMAP binaries as Lab 6. gsplat is installed from PyPI — no compilation required on Colab.

In [ ]:
# ── Video ingestion
!pip install yt-dlp -q
!apt-get install -y ffmpeg -q

# ── Pre-built COLMAP + GLOMAP binaries (same as Lab 6)
!wget -q https://github.com/nickmccarty/3dgs-workshop/raw/refs/heads/master/colmap.zip
!wget -q https://github.com/nickmccarty/3dgs-workshop/raw/refs/heads/master/glomap-1.0.0.zip
!unzip -q colmap.zip
!unzip -q glomap-1.0.0.zip

# ── gsplat (MIT licensed)
!pip install gsplat -q

# ── gsplat training scripts + evaluation dependencies
!git clone --depth 1 https://github.com/nerfstudio-project/gsplat.git gsplat_repo -q
!pip install -r gsplat_repo/examples/requirements.txt -q

# ── PLY parsing
!pip install plyfile -q

print('All dependencies installed.')

In [ ]:
import os
os.environ['PATH'] += ':/content/colmap/build/src/colmap/exe'
os.environ['PATH'] += ':/content/glomap-1.0.0/build/glomap'

!colmap -h 2>&1 | head -3
!glomap -h 2>&1 | head -3

## §1 — Pipeline Timer

Same timing harness as Lab 6 — compare your elapsed times against the reference to benchmark gsplat vs the original training loop.

In [ ]:
import time
import contextlib
import pandas as pd

_STAGES = []

@contextlib.contextmanager
def stage(name, expected_output=''):
    print(f'\n▶ {name}...')
    t0 = time.time()
    yield
    elapsed = time.time() - t0
    _STAGES.append({'Stage': name, 'Elapsed (s)': round(elapsed, 1), 'Expected output': expected_output})
    print(f'  ✓ Done in {elapsed:.1f}s')

def timing_summary():
    df = pd.DataFrame(_STAGES)
    df['Elapsed (min)'] = (df['Elapsed (s)'] / 60).round(1)
    display(df)
    print(f'Total: {df["Elapsed (s)"].sum() / 60:.1f} min')

## §2 — Video Download & Frame Extraction

We download the same Bamburgh Castle fly-through used in Lab 6 and extract frames at **7 fps** with ffmpeg.
At 7 fps the 2:27 video yields ~1,031 frames.

In [ ]:
from yt_dlp import YoutubeDL

URL = 'https://youtu.be/nefKdNDJR9M?si=4PDguQYUWTdTtZtG'

with stage('Video download', 'downloaded_video.mp4 (~6 MB)'):
    ydl_opts = {'format': 'best', 'outtmpl': './downloaded_video.%(ext)s', 'quiet': True}
    with YoutubeDL(ydl_opts) as ydl:
        ydl.download([URL])

In [ ]:
from IPython.display import Video
Video('downloaded_video.mp4', width=640, embed=True)

In [ ]:
with stage('Frame extraction — 7 fps', 'data/images/ — ~1,031 frames @ 640×272'):
    %cd data
    !mkdir -p images
    !ffmpeg -i ../downloaded_video.mp4 -qscale:v 1 -qmin 1 -vf "fps=7" images/image%04d.jpg -y -loglevel error

import os
n_frames = len([f for f in os.listdir('images') if f.endswith('.jpg')])
print(f'Extracted {n_frames} frames')

## §3 — COLMAP: Feature Extraction & Matching

COLMAP detects SIFT keypoints in every frame and runs exhaustive matching to find correspondences.
These correspondences seed GLOMAP's global reconstruction in §4.

In [ ]:
with stage('COLMAP feature extraction', 'database.db with SIFT keypoints'):
    !colmap feature_extractor \
       --database_path database.db \
       --image_path images \
       --ImageReader.single_camera 1 \
       --SiftExtraction.use_gpu 1 \
       --SiftExtraction.num_threads 4

In [ ]:
with stage('COLMAP exhaustive matching', '~96k verified image pairs in database.db'):
    !colmap exhaustive_matcher \
       --database_path database.db \
       --SiftMatching.use_gpu 1

## §4 — GLOMAP: Global Reconstruction

GLOMAP runs five stages (view graph calibration → relative pose estimation → rotation averaging →
global positioning → bundle adjustment) and writes COLMAP-compatible binary output to `sparse/0/`.
This is exactly the format gsplat's `simple_trainer.py` expects.

In [ ]:
with stage('GLOMAP global reconstruction', 'sparse/0/ — cameras.bin, images.bin, points3D.bin'):
    !glomap mapper \
        --database_path database.db \
        --image_path images \
        --output_path sparse

In [ ]:
%cd ..

## §5 — SfM Output Inspection

Parse `images.bin` to count registered cameras and visualize the reconstructed camera trajectory.

In [ ]:
import struct
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

def read_images_binary(path):
    images = {}
    with open(path, 'rb') as f:
        n = struct.unpack('<Q', f.read(8))[0]
        for _ in range(n):
            img_id = struct.unpack('<I', f.read(4))[0]
            qvec = struct.unpack('<4d', f.read(32))
            tvec = struct.unpack('<3d', f.read(24))
            cam_id = struct.unpack('<I', f.read(4))[0]
            name = b''
            while True:
                c = f.read(1)
                if c == b'\x00': break
                name += c
            n_pts = struct.unpack('<Q', f.read(8))[0]
            f.read(24 * n_pts)  # skip point2D data
            images[img_id] = {'qvec': qvec, 'tvec': tvec, 'name': name.decode()}
    return images

images = read_images_binary('data/sparse/0/images.bin')
print(f'Registered cameras: {len(images)} / ~1,031 frames')

# Plot camera centers
centers = []
for img in images.values():
    q = np.array(img['qvec'])
    t = np.array(img['tvec'])
    R = np.array([
        [1-2*(q[2]**2+q[3]**2), 2*(q[1]*q[2]-q[0]*q[3]), 2*(q[1]*q[3]+q[0]*q[2])],
        [2*(q[1]*q[2]+q[0]*q[3]), 1-2*(q[1]**2+q[3]**2), 2*(q[2]*q[3]-q[0]*q[1])],
        [2*(q[1]*q[3]-q[0]*q[2]), 2*(q[2]*q[3]+q[0]*q[1]), 1-2*(q[1]**2+q[2]**2)]
    ])
    centers.append(-R.T @ t)

centers = np.array(centers)
fig = plt.figure(figsize=(8, 5))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(centers[:,0], centers[:,1], centers[:,2], s=1, c='steelblue', alpha=0.6)
ax.set_title('Registered camera centers — Bamburgh Castle')
ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
plt.tight_layout()
plt.show()

## §6 — gsplat Training

gsplat's `simple_trainer.py` takes a COLMAP-format directory and trains a 3DGS model for 30,000 iterations.
It evaluates PSNR, SSIM, and LPIPS at configurable checkpoints and writes results to a JSON file.

**Note on the `--data_factor` flag**: setting `--data_factor 1` uses images at full resolution (640×272).
Increase to `2` or `4` to downsample and train faster if you run out of VRAM.

In [ ]:
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
with stage('gsplat training — 30,000 iterations', 'results/bamburgh/ply/point_cloud.ply'):
    !python gsplat_repo/examples/simple_trainer.py default \
        --data_dir /content/data \
        --data_factor 1 \
        --result_dir /content/results/bamburgh \
        --max_steps 30000 \
        --eval_steps 7000 15000 30000 \
        --save_steps 30000 \
        --disable_viewer

## §7 — Evaluation: PSNR, SSIM, LPIPS

gsplat logs per-checkpoint metrics to a JSON file. We parse and display them here.

**Reference values** (Lab 6 baseline with the original `gaussian-splatting` on Bamburgh Castle):
- PSNR: ~29.6 dB (30k iterations, no depth priors)
- SSIM: ~0.88
- LPIPS: ~0.18

gsplat should reach comparable numbers. Differences arise from minor implementation choices (opacity reset schedule, densification thresholds) and the Colab GPU allocation.

In [ ]:
import json
import glob

# gsplat writes stats to a JSON file in the result directory
stats_paths = glob.glob('/content/results/bamburgh/stats/*.json')
if not stats_paths:
    stats_paths = glob.glob('/content/results/bamburgh/**/*.json', recursive=True)

all_stats = []
for p in sorted(stats_paths):
    with open(p) as f:
        data = json.load(f)
    if isinstance(data, list):
        all_stats.extend(data)
    elif isinstance(data, dict):
        all_stats.append(data)

if all_stats:
    df_eval = pd.DataFrame(all_stats)
    metric_cols = [c for c in df_eval.columns if any(m in c.lower() for m in ['psnr', 'ssim', 'lpips', 'step'])]
    display(df_eval[metric_cols] if metric_cols else df_eval)
else:
    print('No stats JSON found — check the result_dir path.')

In [ ]:
# ── Plot PSNR over training steps
import matplotlib.pyplot as plt

if all_stats and 'psnr' in pd.DataFrame(all_stats).columns:
    df_plot = pd.DataFrame(all_stats).sort_values('step')
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    for ax, metric in zip(axes, ['psnr', 'ssim', 'lpips']):
        if metric in df_plot.columns:
            ax.plot(df_plot['step'], df_plot[metric], marker='o')
            ax.set_title(metric.upper())
            ax.set_xlabel('Step')
            ax.grid(alpha=0.3)
    plt.suptitle('gsplat — Bamburgh Castle evaluation metrics', fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    print('Metric columns not found — inspect all_stats for key names.')

## §8 — PLY Analysis

Confirm that gsplat outputs the same 59-property Gaussian schema as the original `gaussian-splatting`.
This verifies downstream `.splat` conversion will work without modification.

In [ ]:
from plyfile import PlyData
import glob

ply_paths = sorted(glob.glob('/content/results/bamburgh/**/*.ply', recursive=True))
assert ply_paths, 'No PLY found — did training complete?'

ply_path = ply_paths[-1]
print(f'Loading: {ply_path}')

plydata = PlyData.read(ply_path)
vertex = plydata['vertex']
props = [p.name for p in vertex.properties]

print(f'\nGaussian count : {len(vertex):,}')
print(f'Properties     : {len(props)}')
print(f'Schema         : {props[:6]} ... {props[-3:]}')

# Verify 59-property schema
expected = {'x','y','z','nx','ny','nz','opacity'}
has_sh   = any('f_dc' in p for p in props)
has_scale = any('scale' in p for p in props)
has_rot   = any('rot' in p for p in props)

print(f'\nSH coefficients : {has_sh}')
print(f'Scale params    : {has_scale}')
print(f'Rotation params : {has_rot}')
print(f'Expected schema : {len(props) == 59}')

## §9 — Export to `.splat`

The `.splat` format packs each Gaussian into 32 bytes: 12B position, 12B scale, 4B RGBA color, 4B
quaternion. Gaussians are sorted by opacity-weighted volume (descending) so renderers can
do a single front-to-back pass.

The conversion function is identical to Lab 6 — gsplat's PLY is schema-compatible.

In [ ]:
import numpy as np
from io import BytesIO

SH_C0 = 0.28209479177387814

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def ply_to_splat(plydata):
    v = plydata['vertex']
    sort_key = -np.exp(
        np.array(v['scale_0']) + np.array(v['scale_1']) + np.array(v['scale_2'])
    ) / (1 + np.exp(-np.array(v['opacity'])))
    idx = np.argsort(sort_key)

    buf = BytesIO()
    for i in idx:
        g = v[i]
        pos    = np.array([g['x'], g['y'], g['z']], dtype=np.float32)
        scales = np.exp(np.array([g['scale_0'], g['scale_1'], g['scale_2']], dtype=np.float32))
        rot    = np.array([g['rot_0'], g['rot_1'], g['rot_2'], g['rot_3']], dtype=np.float32)
        color  = np.array([
            0.5 + SH_C0 * g['f_dc_0'],
            0.5 + SH_C0 * g['f_dc_1'],
            0.5 + SH_C0 * g['f_dc_2'],
            sigmoid(g['opacity'])
        ])
        buf.write(pos.tobytes())
        buf.write(scales.tobytes())
        buf.write((color * 255).clip(0, 255).astype(np.uint8).tobytes())
        buf.write(((rot / np.linalg.norm(rot)) * 128 + 128).clip(0, 255).astype(np.uint8).tobytes())
    return buf.getvalue()

with stage('PLY → .splat conversion', 'bamburgh_gsplat.splat'):
    splat_bytes = ply_to_splat(plydata)
    splat_path  = '/content/bamburgh_gsplat.splat'
    with open(splat_path, 'wb') as f:
        f.write(splat_bytes)

import os
n_gaussians = len(plydata['vertex'])
splat_mb    = os.path.getsize(splat_path) / 1e6
ply_mb      = os.path.getsize(ply_path)   / 1e6
print(f'Gaussians  : {n_gaussians:,}')
print(f'PLY size   : {ply_mb:.1f} MB  ({n_gaussians * len(plydata["vertex"].properties) * 4 / 1e6:.1f} MB uncompressed)')
print(f'.splat size: {splat_mb:.1f} MB  ({n_gaussians * 32 / 1e6:.1f} MB theoretical)')

## §10 — Pipeline Timing Summary

In [ ]:
timing_summary()

## §11 — WebGL Viewer

Download `bamburgh_gsplat.splat` from the Colab file browser and drag it into one of:

- **[antimatter15's viewer](https://antimatter15.com/splat/)** — most widely used
- **[SuperSplat](https://playcanvas.com/supersplat/editor)** — supports editing and export
- **[3DGS.app](https://3dgs.app)** — mobile-friendly

The `.splat` file produced here is identical in format to the one produced in Lab 6 —
any viewer that supports the original format will work with gsplat output.

In [ ]:
from google.colab import files
files.download(splat_path)

---
## §12 — License Comparison: gsplat vs gaussian-splatting

Now that you have run both pipelines (Lab 6 and Lab 6b), compare what the license difference means in practice:

**Discussion questions**

1. Both notebooks produced a `.splat` file from the same source video. At what point in the pipeline did the license distinction actually matter?

2. The `utils/loss_utils.py` MIT carve-out in the original `gaussian-splatting` repository only covers one file. If you were building a commercial reconstruction service and wanted to use the original Inria code for everything *except* the training loop, would that be sufficient? Why or why not?

3. gsplat's `simple_trainer.py` adds built-in PSNR/SSIM/LPIPS evaluation that the original `gaussian-splatting` does not include. How does this change the development workflow for production systems compared to Lab 6?

4. Suppose a client asks you to deploy a 3DGS reconstruction service and explicitly says they do not want to contact Inria. List three technically viable paths forward that do not involve the original `gaussian-splatting` repository.

---
## Lab 6b Completion Checklist

- [ ] All pipeline stages completed; timing DataFrame produced
- [ ] GLOMAP registered > 500 cameras from the Bamburgh Castle video
- [ ] gsplat training reached 30,000 iterations without OOM errors
- [ ] PSNR at step 30,000 is within 1 dB of the Lab 6 reference (29.6 dB)
- [ ] PLY confirmed as 59-property schema, compatible with `.splat` conversion
- [ ] `.splat` file downloaded and rendered in a WebGL viewer
- [ ] Discussion questions in §12 answered